# Reinforcement Learning with PPO on Ray RLlib
In this notebook, we train a Proximal Policy Optimization (PPO) agent with **Ray RLlib** instead of writing the algorithm by hand. Unlike standard Supervised Learning where we fit a model to a fixed dataset, here we train an agent to interact with a dynamic environment (a building simulation).

## Learning Objective
- **Write a Gymnasium environment**: Package a physics simulation behind the standard `reset()` / `step()` contract so that any RL library can drive it.
- **Configure PPO declaratively**: Express clipping, entropy bonus, value loss weighting, GAE, and minibatching as a `PPOConfig` rather than as a custom `train_step`.
- **Understand what the library does for you**: We still derive GAE and the clipped surrogate loss by hand on paper, so you can read RLlib's metrics and know exactly what they mean.
- **Control Physical Systems**: Train an agent to optimize a physical system using real-world data, then export and serve it.

### What changed relative to the from-scratch version
| Hand-written before | Provided by RLlib now |
|---|---|
| Subclassed `keras.Model` with actor + critic | `DefaultPPOTorchRLModule`, configured via `DefaultModelConfig` |
| `gaussian_log_likelihood`, `ppo_loss`, `entropy` | PPO `Learner` loss |
| `calculate_gae` in the rollout loop | `GeneralizedAdvantageEstimation` connector |
| Manual rollout buffer + `model.fit()` | `EnvRunner` actors + `algo.train()` |
| `KLEarlyStopping` callback | Adaptive KL coefficient (`kl_coeff`, `kl_target`) |

What does **not** change: the environment, the physics, the reward function, and the intuition behind advantages. Those are still yours to design, and they are still where almost all the difficulty lives.

> **Tested with Ray 2.47.1** (new API stack, PyTorch), which pins `gymnasium==1.0.0`. Every config argument used here is available in 2.47.1 — the generic ones (`num_epochs`, `minibatch_size`, `train_batch_size_per_learner`, `grad_clip_by`) reach `PPOConfig.training()` through its `**kwargs` and land on the base `AlgorithmConfig`. On releases older than 2.47 several of these still carry their pre-rename names (`num_sgd_iter`, `sgd_minibatch_size`), so the setup cell asserts the version rather than letting the mismatch surface halfway through a training run.


## Setup

In [ ]:
# This notebook targets Ray 2.47.1 on the new API stack.
# Ray 2.47.1 pins gymnasium==1.0.0, so let pip resolve it rather than pinning both.
# !pip install -q "ray[rllib]==2.47.1" torch pandas matplotlib

import os
import urllib.request
import warnings

from ray.util.annotations import RayDeprecationWarning

warnings.filterwarnings("ignore", category=RayDeprecationWarning)

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from gymnasium import spaces

import ray
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.callbacks.callbacks import RLlibCallback
from ray.rllib.core import DEFAULT_MODULE_ID
from ray.rllib.core.columns import Columns
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from ray.rllib.core.rl_module.rl_module import RLModule
from ray.rllib.utils.metrics import (
    ENV_RUNNER_RESULTS,
    EPISODE_RETURN_MEAN,
    LEARNER_RESULTS,
    NUM_ENV_STEPS_SAMPLED_LIFETIME,
)
from ray.tune.registry import register_env

print(f"Ray:       {ray.__version__}")
print(f"PyTorch:   {torch.__version__}")
print(f"Gymnasium: {gym.__version__}")

# Fail loudly rather than midway through training if the API stack has moved.
_major, _minor, *_ = (int(x) for x in ray.__version__.split(".")[:2])
assert (_major, _minor) >= (2, 47), (
    f"This notebook targets Ray >= 2.47 (tested on 2.47.1); found {ray.__version__}. "
    "Earlier releases use the pre-rename config arguments "
    "(num_sgd_iter / sgd_minibatch_size instead of num_epochs / minibatch_size)."
)

## Loading and Normalizing Data

We are building a control system. In Control Theory terms, the Outdoor Temperature acts as an External Disturbance—a force trying to push our system away from its target state. <br>
We use the [**Jena Climate Dataset**](https://www.kaggle.com/datasets/mnassrib/jena-climate), which is sampled every 10 minutes, to drive our simulation.

In [ ]:
uri = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
zip_path = "jena_climate_2009_2016.csv.zip"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(uri, zip_path)

df = pd.read_csv(zip_path)
df.head()

Let's split the data into training and test.

In [ ]:
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

Neural networks struggle with unscaled data, so let's normalize the data.

In [ ]:
# Calculate stats for normalization
temp_mean = train_df["T (degC)"].mean()
temp_std = train_df["T (degC)"].std()
pres_mean = train_df["p (mbar)"].mean()
pres_std = train_df["p (mbar)"].std()

# Normalize features


def normalize_data(df):
    return np.column_stack(
        [
            df["T (degC)"].values,  # Raw Outdoor
            (df["T (degC)"].values - temp_mean) / temp_std,  # Norm Outdoor
            (df["p (mbar)"].values - pres_mean) / pres_std,  # Norm Pressure
        ]
    ).astype(np.float32)


train_data_norm = normalize_data(train_df)
test_data_norm = normalize_data(test_df)

In [ ]:
plt.title("Normalized Hourly Temperature")
plt.plot(train_data_norm[:, 1])
plt.show()

## Defining the Environment

In Reinforcement Learning, the "Environment" is the world the agent lives in. We build a custom Physics Simulation class (`BuildingControlEnv`) to represent a room's thermodynamics.

The physics and the reward are identical to the from-scratch version. What changes is the **interface**: the class now subclasses `gymnasium.Env` and follows the standard contract, which is what lets RLlib run many copies of it in parallel worker processes without knowing anything about buildings.

### The Physics Model

$$Temp_{next} = Temp_{current} + \underbrace{k \cdot (Temp_{outdoor} - Temp_{current})}_{\text{Passive Heat Loss}} + \underbrace{\text{Action} \cdot \text{HvacMaxPower}}_{\text{Active HVAC}}$$

- Insulation Factor ($k=0.1$): Represents how "leaky" the room is. A higher value means the room temperature matches the outdoors faster.
- HVAC Power: The agent's chosen action (scaled by `HVAC_POWER`) adds or removes heat.

### The Reward Function (The Objective)

Designing the reward function is the most critical part of RL. We want the agent to balance two conflicting goals:
- Comfort: Keep the temperature at 22°C.
- Efficiency: Minimize electricity usage.

### The Gymnasium contract

Four things are required, and each one differs slightly from the hand-rolled version:

1. **`observation_space` / `action_space`** — RLlib inspects these to size the network and pick the action distribution. A `Box` action space with `shape=(1,)` is what makes PPO use a diagonal Gaussian policy; if we had declared `Discrete(3)`, we would get a categorical policy and the exact same config would still work.
2. **`__init__(self, config)`** — RLlib constructs environments from a single config dict, which it ships to each worker process. That is why the weather array is passed in through `env_config` rather than captured from a global.
3. **`reset()` returns `(observation, info)`** and accepts a `seed`. Use `self.np_random` (seeded by `super().reset(seed=seed)`) for randomness, so parallel workers explore different slices of the year instead of all replaying the same week.
4. **`step()` returns 5 values**: `(obs, reward, terminated, truncated, info)`. The split matters for correctness. `terminated` means the episode genuinely ended (the pole fell, the game was lost) and future value is truly zero. `truncated` means we cut a still-viable episode short at the time limit, and the value function should still bootstrap from the final state. Our building never "fails" — it just reaches the end of the rollout window — so we always return `terminated=False, truncated=True`. Collapsing these into one `done` flag, as the original did, silently teaches the critic that the world ends every 1008 steps.

Observations must be plain NumPy arrays of the declared dtype (`float32`), not backend tensors.

In [ ]:
# ==============================================================================
# Environment Configuration & Constants
# ==============================================================================
TARGET_TEMP = 22.0       # Comfortable indoor temperature target (°C)
HVAC_POWER = 5.0         # Maximum temperature change from HVAC in one step (°C)
INSULATION_FACTOR = 0.1  # Thermal insulation coefficient (passive heat transfer)
REWARD_SCALE = 0.1       # Scale factor for comfort penalties
ENERGY_COST = 0.1        # Penalty multiplier for energy consumption (action squared)

# Explicit index mapping for time-series data columns to ensure self-documentation
COL_OUTDOOR_TEMP = 0     # Index 0: Raw outdoor temperature (°C)
COL_NORM_OUTDOOR = 1     # Index 1: Normalized outdoor temperature (for state observation)
COL_NORM_PRESSURE = 2    # Index 2: Normalized atmospheric pressure (for state observation)


class BuildingControlEnv(gym.Env):
    """
    A simulated Gymnasium environment for smart building thermal control.

    The environment models a building's indoor temperature dynamics affected by
    ambient outdoor temperature (passive heat transfer) and an HVAC system (active
    control). The goal is to maintain thermal comfort while minimizing energy use.

    Observation (Box, shape (4,)):
        [Norm_Indoor_Temp, Norm_Outdoor_Temp, Norm_Pressure, Last_Action]
    Action (Box, shape (1,), range [-1, 1]):
        Continuous HVAC effort. Negative cools, positive heats.
    """

    metadata = {"render_modes": []}

    def __init__(self, config=None):
        """
        Args:
            config: Dict shipped by RLlib to every worker. Keys:
                data:      (N, 3) array of [OutdoorTemp, NormOutdoorTemp, NormPressure].
                max_steps: Episode length in timesteps.
                temp_mean: Mean used to normalize the indoor temperature.
                temp_std:  Std used to normalize the indoor temperature.
                start_idx: Optional fixed start index (useful for evaluation).
        """
        super().__init__()
        config = config or {}

        self.data = np.asarray(config["data"], dtype=np.float32)
        self.max_steps = int(config.get("max_steps", 720))
        self.temp_mean = float(config.get("temp_mean", 20.0))
        self.temp_std = float(config.get("temp_std", 10.0))
        self.start_idx = config.get("start_idx", None)

        # RLlib reads these to build the network and choose the action distribution.
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(4,), dtype=np.float32
        )
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(1,), dtype=np.float32
        )

    def _get_obs(self):
        """Constructs the normalized observation vector (shape (4,), float32)."""
        indoor_norm = (self.indoor - self.temp_mean) / self.temp_std
        row = self.data[self.idx]

        return np.array(
            [
                indoor_norm,
                row[COL_NORM_OUTDOOR],
                row[COL_NORM_PRESSURE],
                self.last_action,
            ],
            dtype=np.float32,
        )

    def reset(self, *, seed=None, options=None):
        """
        Resets the environment to begin a new episode.

        Returns:
            (observation, info) — the Gymnasium two-tuple.
        """
        # Seeds self.np_random so each parallel worker samples a different week.
        super().reset(seed=seed)

        idx = (options or {}).get("start_idx", self.start_idx)
        if idx is None:
            # Pick a safe random start, leaving room for a full episode.
            idx = self.np_random.integers(0, len(self.data) - self.max_steps - 1)

        self.idx = int(idx)
        self.step_cnt = 0
        self.indoor = 20.0      # Start slightly cool (°C)
        self.last_action = 0.0  # Clear the action history

        return self._get_obs(), {}

    def _update_physics(self, action):
        """Newton's law of cooling for passive transfer, plus active HVAC injection."""
        outdoor = float(self.data[self.idx][COL_OUTDOOR_TEMP])

        # 1. Passive heat transfer: heat flows from hot to cold.
        self.indoor -= INSULATION_FACTOR * (self.indoor - outdoor)

        # 2. Active thermal control: HVAC power injection.
        self.indoor += action * HVAC_POWER

    def _compute_reward(self, action):
        """Quadratic comfort penalty (clipped) plus quadratic energy penalty."""
        error = self.indoor - TARGET_TEMP
        comfort_penalty = -REWARD_SCALE * (error ** 2)

        # Clip to prevent extreme gradients during early training.
        comfort_penalty = max(comfort_penalty, -1.0)

        # Quadratic cost discourages high-amplitude oscillation.
        energy_penalty = -ENERGY_COST * (action ** 2)

        return comfort_penalty + energy_penalty

    def step(self, action):
        """
        Advances the simulation by one timestep.

        Returns:
            (obs, reward, terminated, truncated, info) — the Gymnasium five-tuple.
        """
        # Actions arrive as arrays of shape (1,); reduce to a bounded scalar.
        action = float(np.clip(np.asarray(action, dtype=np.float32).reshape(-1)[0], -1.0, 1.0))

        self._update_physics(action)
        reward = self._compute_reward(action)

        self.last_action = action
        self.idx += 1
        self.step_cnt += 1

        # The building never "fails", so the episode is only ever time-limited.
        # terminated=False tells the critic to keep bootstrapping past the cutoff.
        terminated = False
        truncated = self.step_cnt >= self.max_steps

        info = {
            "raw_indoor_temp": self.indoor,
            "step_count": self.step_cnt,
        }

        return self._get_obs(), float(reward), terminated, truncated, info

Before handing the environment to a distributed trainer, run Gymnasium's checker. It catches dtype mismatches, wrong tuple arity, and non-reproducible seeding — failures that are painful to diagnose once they surface inside a remote worker.

We then register the environment under a string name. RLlib serializes that name plus `env_config` to every worker, which constructs its own copy.

In [ ]:
from gymnasium.utils.env_checker import check_env

# Smoke-test with a small slice of the training data.
_probe = BuildingControlEnv({"data": train_data_norm[:5000], "max_steps": 100})
check_env(_probe, skip_render_check=True)
print("Environment passes the Gymnasium API check.")
print(f"Observation space: {_probe.observation_space}")
print(f"Action space:      {_probe.action_space}")

# Register under a name so RLlib workers can build their own instances.
register_env("BuildingControl-v0", lambda config: BuildingControlEnv(config))

## Configuring PPO with RLlib

### Actor-Critic Network
PPO needs a network that does two very different jobs at once — the Actor-Critic architecture.

- The Actor (The "Doer"):
  - Goal: Learn what to do (the Policy $\pi$).
  - Input: The observation (indoor temp, outdoor temp, pressure, last action).
  - Output: A Gaussian distribution defined by mean ($\mu$) and $\log \sigma$.
- The Critic (The "Judge"):
  - Goal: Learn how good a state is (the Value Function $V(s)$).
  - Output: A single number representing the expected sum of future rewards.
  - Role: The Critic's prediction is the "baseline." If the Actor gets a reward higher than the Critic predicted, the action was good (positive Advantage).

Instead of subclassing a model, we describe the architecture with `DefaultModelConfig` and RLlib assembles a `DefaultPPOTorchRLModule` from it:

- `fcnet_hiddens=[64, 64]`, `fcnet_activation="tanh"` — two hidden layers per encoder.
- `vf_share_layers=False` — actor and critic get **separate** encoders, matching the two independent `Sequential` stacks in the from-scratch version. Leaving this at its default (`True`) makes the policy and value gradients fight over one trunk, which usually needs `vf_loss_coeff` retuning.
- `free_log_std=True` — this is the direct equivalent of `self.add_weight(name="log_std", ...)`. The standard deviation becomes a single state-independent trainable parameter rather than a network output. Without it, $\log\sigma$ is predicted per-state, and early in training the policy can collapse its own exploration noise before it has learned anything.

One thing we lose: the original used ReLU on the first layer and tanh on the second. `DefaultModelConfig` takes a single activation for the stack. Mixed activations would require a custom `RLModule` — a reasonable next exercise, but not worth it here.

### Loss Functions
RLlib computes these for you, but you should still be able to read them.

#### Actor Loss
$$L^{CLIP}(\theta) = \hat{\mathbb{E}}_t [\min(r_t(\theta)\hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t)]$$
- The Ratio ($r_t$): How much the new policy differs from the old one.
- The Advantage ($\hat{A}_t$): Did the action beat expectations?
- The Clip ($\epsilon = 0.2$, our `clip_param`): The safety brake. Changes that shift the probability ratio by more than 20% are ignored.

#### Critic Loss
The critic minimizes squared error against the GAE returns, scaled by `vf_loss_coeff`.

#### Entropy
An exploration bonus, weighted by `entropy_coeff`, added with a negative sign so that maximizing entropy minimizes loss.

### Mapping the hyperparameters

| From-scratch | RLlib | Value |
|---|---|---|
| `CLIP_RATIO` | `clip_param` | 0.2 |
| `ENTROPY_WEIGHT` | `entropy_coeff` | 0.01 |
| `CRITIC_WEIGHT` | `vf_loss_coeff` | 0.5 |
| `GAMMA` / `LAMBDA` | `gamma` / `lambda_` | 0.99 / 0.95 |
| `LR` | `lr` | 3e-4 |
| `global_clipnorm=0.5` | `grad_clip=0.5`, `grad_clip_by="global_norm"` | — |
| `STEPS_PER_ROLLOUT` | `train_batch_size_per_learner` | 1008 |
| `BATCH_SIZE` | `minibatch_size` | 32 |
| `MAX_EPOCHS_PER_ROLLOUT` | `num_epochs` | 10 (see note) |
| `TARGET_KL` + `KLEarlyStopping` | `use_kl_loss`, `kl_coeff`, `kl_target` | 0.025 |

**Two notes on that last pair of rows.**

The original allowed up to 50 epochs per rollout and relied on the KL callback to abort early — which it usually did, well before 50. RLlib has no mid-`train()` abort hook, so 50 epochs would mean 50 full epochs (over 1500 gradient steps on 1008 samples), which overfits each batch badly. We set `num_epochs=10` instead, which is closer to what the early-stopping version actually executed.

The replacement safety mechanism is different in kind, not just in name. Hard early stopping is reactive: it lets a destructive update land, notices KL has blown past the target, and halts. RLlib's adaptive KL is *preventive* — it adds $\beta \cdot D_{KL}(\pi_{old} \| \pi_{new})$ to the loss and doubles or halves $\beta$ between iterations depending on whether measured KL overshot `kl_target`. Divergence is penalized in the gradient itself. Watch `curr_kl_coeff` climb in the logs when the policy starts moving too fast.

### Parallelism, and one sharp edge
`num_env_runners=2` spawns two worker processes, each running its own copy of the building. This is the concrete payoff of the Gymnasium refactor: sampling scales with cores, and each worker starts from a different week of weather, which decorrelates the batch.

The sharp edge: each `EnvRunner` is a Ray actor that reserves **its own CPU**. If you request more runners than the machine has cores, Ray does not error or fall back — it queues the actors and waits, and `algo.train()` hangs indefinitely with no output. On a 1- or 2-vCPU instance this looks exactly like a very slow first iteration. We therefore derive the count from `os.cpu_count()` below; `0` is a legitimate setting that samples in the driver process, and it is what you want when debugging anyway, since exceptions surface directly instead of inside a remote actor.

In [ ]:
# ==============================================================================
# PPO Hyperparameters & Configuration
# ==============================================================================
STEPS_PER_ROLLOUT = 1008   # Rollout buffer size per iteration (1008 steps = exactly 1 week)
BATCH_SIZE = 32            # Mini-batch size for each optimizer step
NUM_EPOCHS = 50            # Passes over each train batch
LR = 0.0003                # Adam learning rate
CLIP_RATIO = 0.2           # PPO trust region (epsilon)
ENTROPY_WEIGHT = 0.01      # Exploration bonus weight
CRITIC_WEIGHT = 0.5        # Value loss scaling
GAMMA = 0.99               # Discount factor
LAMBDA = 0.95              # GAE smoothing
TARGET_KL = 0.025          # Adaptive KL penalty target

# Each EnvRunner is a separate Ray actor that needs its own CPU. Asking for more
# runners than the machine has cores makes Ray wait forever for resources that
# never free up, so cap this by the actual core count. 0 means "sample in the
# driver process" — slower, but it always runs and is far easier to debug.
NUM_ENV_RUNNERS = 0 #min(2, max(0, (os.cpu_count() or 1) - 1))
print(f"Detected {os.cpu_count()} CPUs -> using num_env_runners={NUM_ENV_RUNNERS}")

env_config = {
    "data": train_data_norm,
    "max_steps": STEPS_PER_ROLLOUT,
    "temp_mean": 20.0,
    "temp_std": 10.0,
}

config = (
    PPOConfig()
    # --- Which environment, and what to pass to each worker's copy of it.
    .environment("BuildingControl-v0", env_config=env_config)
    .framework("torch")
    # --- Sampling: two parallel worker processes collecting experience.
    .env_runners(num_env_runners=NUM_ENV_RUNNERS, rollout_fragment_length="auto")
    # --- Learning: a single local learner (set num_learners>0 for multi-GPU).
    .learners(num_learners=0)
    .training(
        gamma=GAMMA,
        lambda_=LAMBDA,
        clip_param=CLIP_RATIO,
        entropy_coeff=ENTROPY_WEIGHT,
        vf_loss_coeff=CRITIC_WEIGHT,
        # Adaptive KL penalty replaces the hand-written early-stopping callback.
        use_kl_loss=True,
        kl_coeff=0.2,
        kl_target=TARGET_KL,
        lr=LR,
        grad_clip=0.5,
        grad_clip_by="global_norm",
        train_batch_size_per_learner=STEPS_PER_ROLLOUT,
        minibatch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
    )
    # --- Network architecture.
    .rl_module(
        model_config=DefaultModelConfig(
            fcnet_hiddens=[64, 64],
            fcnet_activation="tanh",
            vf_share_layers=False,  # separate actor and critic encoders
            free_log_std=True,      # state-independent log_std, as before
        )
    )
    .debugging(log_level="ERROR")
)

print("PPO configured.")

## Generalized Advantage Estimation (GAE)

RLlib computes GAE for you, inside a connector that runs between sampling and learning. You never call it. But `gamma` and `lambda_` are two of the most consequential knobs in the config, so it is worth keeping the intuition sharp.

In Reinforcement Learning, looking at a single reward isn't enough. We need to know if a good result was just luck (noise) or part of a winning streak (signal). **GAE** combines two pieces of information to measure this "momentum":

**The Inputs (What we know)**
- `values` (The Expectation): What the Critic predicted would happen.
  - Analogy: "I expect to get a 70 on this exam."
- `rewards` (The Reality): What actually happened in the environment.
  - Analogy: "I actually got a 90."
- `next_value` (The Horizon): The Critic's prediction for the state after the current batch ends. This connects our current experience to the future.

**The Outputs (What we use in training)**
- `advantages` (The Bonus $\rightarrow$ Trains the Actor): How much better was the action than expected?
  - Meaning: If positive, the agent performed above baseline. Do this action more.
- `returns` (The Truth $\rightarrow$ Trains the Critic): What was the actual total value of the state, in hindsight?
  - Formula: Advantage + Value.

**The Calculation Logic**
- **Calculate Delta ($\delta$)**: The immediate surprise for one step.
  - `Delta = (Reward + Future Value) - Current Value`
- **Calculate Advantage**: The rolling surprise, accumulated backwards through time.

Below is a reference implementation, kept for illustration only — it is not used in training.

In [ ]:
def gae(rewards, values, next_value, gamma=GAMMA, lam=LAMBDA):
    """Reference NumPy implementation of GAE. RLlib does this internally."""
    rewards = np.asarray(rewards, dtype=np.float32)
    values = np.asarray(values, dtype=np.float32)

    advantages = np.zeros_like(rewards)
    last_gae = 0.0

    next_values = np.concatenate((values[1:], [next_value]), axis=0)

    # Bellman Error (Deltas) = r + gamma * V(next) - V(current)
    deltas = rewards + (gamma * next_values) - values

    decay = gamma * lam
    # Iterate backwards
    for t in reversed(range(len(rewards))):
        # Recursive GAE Formula
        advantages[t] = last_gae = deltas[t] + decay * last_gae

    returns = advantages + values
    return advantages, returns

Let's check how GAE works using an example:
- Scenario: You have a goose.
- Expectation (Values): You think it lays regular eggs (Worth \$2).
- Reality (Rewards): It actually lays GOLDEN eggs (Worth \$10).

In [ ]:
rewards = [10.0, 10.0, 10.0, 10.0, 10.0]  # Reality: It pays $10 daily
values = [2.0, 2.0, 2.0, 2.0, 2.0]  # Expectation: We expected only $2
next_value = 2.0  # Future expectation is also $2

advantages, returns = gae(rewards, values, next_value)

print(f"Calculated Advantage (GAE): \n{np.round(advantages, 2)}")
print(f"\nCorrected Truth (Returns):  \n{np.round(returns, 2)}")

print(
    f"\nDay 1 Return ({returns[0]:.2f}):    The goose is actually worth ${returns[0]:.2f} total."
)
print(
    f"Day 1 Advantage ({advantages[0]:.2f}): But since you expected ${values[0]}, the 'Good Surprise' is only ${advantages[0]:.2f}."
)

## PPO Training

RLlib collapses the rollout / process / update cycle into a single `algo.train()` call. We still write an outer loop, but only to control how many iterations run and to record metrics.

### Custom Callback

The from-scratch version used a Keras callback to watch KL divergence and set `stop_training = True` mid-fit. RLlib's `RLlibCallback` fires at coarser boundaries — `on_train_result` runs after a full `train()` iteration, not between epochs — so the same trick isn't available.

That's fine, because the adaptive KL coefficient is already doing the protective work inside the loss. What we build instead is a **monitor**: it reads the KL that RLlib measured and warns when the policy is moving faster than `kl_target`, so we can see the adaptive penalty kicking in rather than guessing.

Callbacks are also where you would add domain-specific logging — mean indoor temperature, comfort violations, energy spend — via `on_episode_end`. That's usually a better use of the hook than trying to recreate early stopping.

In [ ]:
# ==============================================================================
# 1. Helper Class: KL Divergence Monitor
# ==============================================================================
class KLMonitor(RLlibCallback):
    """
    Reports when the measured policy KL divergence exceeds the target.

    Unlike the Keras `KLEarlyStopping` callback, this does not halt anything:
    RLlib's adaptive `kl_coeff` already penalizes divergence inside the loss.
    This makes that mechanism visible.
    """

    def __init__(self, target_kl=TARGET_KL):
        super().__init__()
        self.target_kl = target_kl

    def on_train_result(self, *, algorithm, metrics_logger=None, result, **kwargs):
        learner = result.get(LEARNER_RESULTS, {}).get(DEFAULT_MODULE_ID, {})
        kl = learner.get("mean_kl_loss")
        coeff = learner.get("curr_kl_coeff")

        if kl is not None and kl > self.target_kl:
            print(
                f"        KL divergence {kl:.4f} exceeded target {self.target_kl:.4f}; "
                f"adaptive penalty now beta={coeff:.3f}"
            )

### The Training Loop

Each call to `algo.train()` runs one full PPO iteration, which is exactly the three phases you wrote by hand before:

#### Phase 1: Data Collection
The `EnvRunner` actors step their environments until `train_batch_size_per_learner` (1008) timesteps have been gathered. PPO is **on-policy**, so this data is used once and discarded. Actions are sampled from the Gaussian for exploration during this phase — `forward_exploration()` — while evaluation later uses the mean.

#### Phase 2: Processing
Connectors compute GAE advantages and returns from the raw trajectories and normalize the advantages.

#### Phase 3: Training
The `Learner` shuffles the batch, splits it into 32-sample minibatches, and runs `num_epochs` passes, applying the clipped surrogate loss, the value loss, the entropy bonus, and the adaptive KL penalty.

The metric to watch is `episode_return_mean`. Since every episode is exactly 1008 steps and per-step reward is bounded below at roughly $-1.1$, the worst possible return is about $-1100$; a well-tuned agent should climb toward zero. Also keep an eye on `vf_explained_var` — if the critic can't explain the returns (values near 0 or negative), the advantages are noise and the actor is learning very little.

**One subtlety about episode metrics.** RLlib reports `episode_return_mean` only for episodes that have *finished*. Our episode length and our train batch are both 1008 steps, so with a single runner one episode completes per iteration and the metric is always present. With `num_env_runners=2`, each runner contributes only 504 steps per iteration, so the first episode does not finish until iteration 2 — and until then the key is **absent from the results dict entirely**, not zero. Indexing it directly raises `KeyError: 'episode_return_mean'`, which is why the loop below uses `.get(..., nan)`. Expect the first row or two of the log to show `nan` when running with multiple runners; the plot simply leaves a gap there.

In [ ]:
%%time
# ==============================================================================
# 2. Main PPO Training Loop
# ==============================================================================
ROLLOUTS = 50  # Total training iterations (each collects STEPS_PER_ROLLOUT steps)

ray.init(ignore_reinit_error=True, log_to_driver=False)

algo = config.callbacks(KLMonitor).build_algo()

print("Starting Training...")

# Training history dictionary for metric visualization
history = {
    "actor": [],
    "critic": [],
    "ent": [],
    "kl": [],
    "rewards": [],
}

for rollout in range(ROLLOUTS):
    result = algo.train()

    env_runner_metrics = result[ENV_RUNNER_RESULTS]
    learner_metrics = result[LEARNER_RESULTS][DEFAULT_MODULE_ID]

    # Episode metrics only exist once an episode has actually FINISHED.
    # Our episodes are STEPS_PER_ROLLOUT steps long, but with N env runners each
    # runner only collects STEPS_PER_ROLLOUT/N steps per iteration -- so with 2
    # runners the first episode completes on iteration 2, and this key is simply
    # missing (not zero) before then. Indexing it directly raises KeyError.
    mean_return = env_runner_metrics.get(EPISODE_RETURN_MEAN, float("nan"))

    history["actor"].append(learner_metrics["policy_loss"])
    history["critic"].append(learner_metrics["vf_loss"])
    history["ent"].append(learner_metrics["entropy"])
    history["kl"].append(learner_metrics["mean_kl_loss"])
    history["rewards"].append(mean_return)

    print(
        f"    Rollout {rollout + 1:02d}/{ROLLOUTS} - "
        f"Mean Episode Return: {history['rewards'][-1]:8.2f} | "
        f"actor: {history['actor'][-1]:7.4f} | "
        f"critic: {history['critic'][-1]:7.4f} | "
        f"entropy: {history['ent'][-1]:6.3f} | "
        f"steps: {int(result[NUM_ENV_STEPS_SAMPLED_LIFETIME]):,}"
    )

print("\nTraining complete.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["rewards"], color="darkgreen")
axes[0].set_title("Mean Episode Return")
axes[0].set_xlabel("Training Iteration")
axes[0].grid(alpha=0.3)

axes[1].plot(history["actor"], label="policy loss")
axes[1].plot(history["critic"], label="value loss")
axes[1].set_title("Losses")
axes[1].set_xlabel("Training Iteration")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(history["ent"], color="purple", label="entropy")
axes[2].plot(history["kl"], color="crimson", label="KL")
axes[2].axhline(TARGET_KL, color="gray", linestyle=":", label="KL target")
axes[2].set_title("Exploration & Policy Drift")
axes[2].set_xlabel("Training Iteration")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluation

After training, we switch the model to inference mode.

During training, actions were sampled from a Gaussian (exploration). For evaluation we use the mean $\mu$ directly (exploitation) to measure the policy's best behavior.

`forward_inference()` is the method that expresses this. It returns `action_dist_inputs` rather than an action — for our one-dimensional Gaussian that tensor has shape `(batch, 2)`, holding $[\mu, \log\sigma]$. Slicing off the first column gives the deterministic action, the exact analogue of taking `mu` from the old three-output `call()`.

We drive the environment directly here rather than through RLlib, so the loop reads much like the original — just with the five-tuple `step()` and a fixed `start_idx` for a reproducible plot.

In [ ]:
TEST_STEPS = 2000

# Pull the trained policy network out of the algorithm.
rl_module = algo.get_module(DEFAULT_MODULE_ID)

# Initialize the evaluation environment on held-out weather.
test_env = BuildingControlEnv(
    {
        "data": test_data_norm,
        "max_steps": TEST_STEPS,
        "temp_mean": 20.0,
        "temp_std": 10.0,
        "start_idx": 0,  # fixed start makes the plot reproducible
    }
)
state, _ = test_env.reset()

# Lists to store logged variables for visualization
indoor_log, outdoor_log, action_log = [], [], []

for t in range(TEST_STEPS):
    # 1. Deterministic forward pass: take the distribution mean, not a sample.
    obs_batch = torch.from_numpy(state).unsqueeze(0)
    with torch.no_grad():
        out = rl_module.forward_inference({Columns.OBS: obs_batch})
    action = float(out[Columns.ACTION_DIST_INPUTS][0, 0])

    # 2. Unpack the five values returned by a Gymnasium step().
    state, reward, terminated, truncated, info = test_env.step(action)

    # 3. Log the physical indoor temperature directly from the environment.
    indoor_log.append(test_env.indoor)

    # 4. Denormalize the outdoor temperature for plotting.
    outdoor_real = (float(state[1]) * temp_std) + temp_mean

    outdoor_log.append(outdoor_real)
    action_log.append(np.clip(action, -1.0, 1.0))

    # Reset if we hit an episode boundary before the test limit.
    if terminated or truncated:
        state, _ = test_env.reset()

print(f"Mean indoor temperature: {np.mean(indoor_log):.2f} °C (target {TARGET_TEMP} °C)")
print(f"Mean absolute error:     {np.mean(np.abs(np.array(indoor_log) - TARGET_TEMP)):.2f} °C")

Let's plot the Indoor Temp controlled by HVAC actions vs. Target Temp to visually verify if the agent learned to anticipate weather changes.

In [ ]:
# Plotting
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel("Hours")
ax1.set_ylabel("Temperature (°C)")
ax1.plot(outdoor_log, color="gray", alpha=0.5, label="Outdoor")
ax1.plot(indoor_log, color="blue", linewidth=2, label="Indoor")
ax1.axhline(y=TARGET_TEMP, color="green", linestyle=":", label="Target")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.set_ylabel("HVAC Power", color="red")
ax2.plot(action_log, color="red", alpha=0.3, label="Action")
ax2.set_ylim(-1.0, 1.0)
ax2.legend(loc="upper right")

plt.title("Smart Thermostat: Evaluation Run (Ray RLlib PPO)")
plt.show()

## Exporting for Serving

This is where the RLlib version diverges most from the Keras one, and the reason is worth understanding.

RLlib produces a **checkpoint** — policy weights plus optimizer state, connector state, and config, designed to *resume training*. It is the wrong artifact for a serving container: it carries training-only baggage, and loading it requires Ray installed in the serving image.

So we do two separate things:

1. **Save the checkpoint** for reproducibility and to resume or fine-tune later.
2. **Distill the policy into a plain `torch.nn.Module`** for serving — a 3-layer MLP holding just the actor weights, with no Ray dependency, exported to TorchScript.

The distillation is exact, not approximate. With `vf_share_layers=False` and `free_log_std=True`, the deterministic action is a straight feed-forward path through `encoder.actor_encoder` and the `pi` head; the critic and the log-std parameter matter only during training. We copy those weights into a bare `Sequential` and assert the outputs match.

We also apply `clamp(-1, 1)` in the exported module, mirroring the clipping the environment does anyway. This makes the served artifact self-contained: the caller can't get a physically meaningless HVAC command out of it.

In [ ]:
import datetime
import shutil

OUTPUT_DIR = "./rl_ppo_export"
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

# Absolute paths matter here: `save_to_path` resolves the destination through
# pyarrow's filesystem layer, which reads the string as a URI and rejects a
# relative path with "URI has empty scheme".
EXPORT_PATH = os.path.abspath(os.path.join(OUTPUT_DIR, TIMESTAMP))
CHECKPOINT_PATH = os.path.join(EXPORT_PATH, "checkpoint")
MODEL_STORE = os.path.join(EXPORT_PATH, "model-store")

os.makedirs(MODEL_STORE, exist_ok=True)

# 1. Full RLlib checkpoint (resume training, restore connectors, etc.)
algo.save_to_path(CHECKPOINT_PATH)
print(f"RLlib checkpoint written to {CHECKPOINT_PATH}")

# The RLModule alone can also be reloaded standalone:
#   RLModule.from_checkpoint(
#       os.path.join(CHECKPOINT_PATH, "learner_group", "learner",
#                    "rl_module", DEFAULT_MODULE_ID)
#   )

In [ ]:
# ==============================================================================
# 2. Distill the actor into a dependency-free TorchScript module
# ==============================================================================
class DeterministicPolicy(torch.nn.Module):
    """Plain MLP reproducing the RLModule's deterministic (mean) action."""

    def __init__(self, obs_dim=4, act_dim=1, hidden=(64, 64)):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(obs_dim, hidden[0]),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden[0], hidden[1]),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden[1], act_dim),
        )

    def forward(self, obs):
        # Clamp mirrors the environment's own action clipping.
        return torch.clamp(self.net(obs), -1.0, 1.0)


state_dict = rl_module.state_dict()
policy = DeterministicPolicy()

# Copy the actor encoder and policy head weights across.
policy.net[0].weight.data = state_dict["encoder.actor_encoder.net.mlp.0.weight"].clone()
policy.net[0].bias.data = state_dict["encoder.actor_encoder.net.mlp.0.bias"].clone()
policy.net[2].weight.data = state_dict["encoder.actor_encoder.net.mlp.2.weight"].clone()
policy.net[2].bias.data = state_dict["encoder.actor_encoder.net.mlp.2.bias"].clone()
policy.net[4].weight.data = state_dict["pi.net.mlp.0.weight"].clone()
policy.net[4].bias.data = state_dict["pi.net.mlp.0.bias"].clone()
policy.eval()

# Verify the distilled network matches the RLModule exactly.
probe = torch.randn(16, 4)
with torch.no_grad():
    reference = rl_module.forward_inference({Columns.OBS: probe})[Columns.ACTION_DIST_INPUTS][:, :1]
    distilled = policy(probe)
max_diff = (torch.clamp(reference, -1.0, 1.0) - distilled).abs().max().item()
assert max_diff < 1e-6, f"Distillation mismatch: {max_diff}"
print(f"Distilled policy matches RLModule (max abs diff {max_diff:.2e})")

# TorchScript, traced with a batch dimension so any batch size serves correctly.
SERIALIZED_FILE = os.path.join(EXPORT_PATH, "policy.pt")
traced = torch.jit.trace(policy, torch.zeros(4, 4))
traced.save(SERIALIZED_FILE)
print(f"TorchScript policy written to {SERIALIZED_FILE}")

### The TorchServe handler

Vertex AI's prebuilt PyTorch containers run TorchServe, which expects a **model archive** (`.mar`) containing the serialized weights plus a handler that defines preprocess → inference → postprocess.

Vertex forwards the request body to the handler, so we receive `{"instances": [[...], ...]}` and must return a list with one entry per instance. Each instance is a 4-element observation.

In [ ]:
handler_code = """
import json

import torch
from ts.torch_handler.base_handler import BaseHandler


# Serves the deterministic HVAC policy. Input: 4-element observations.
class BuildingControlHandler(BaseHandler):

    def preprocess(self, data):
        instances = []
        for row in data:
            payload = row.get("body") or row.get("data")
            if isinstance(payload, (bytes, bytearray, str)):
                payload = json.loads(payload)
            if isinstance(payload, dict):
                instances.extend(payload.get("instances", []))
            else:
                instances.append(payload)
        return torch.tensor(instances, dtype=torch.float32)

    def inference(self, data, *args, **kwargs):
        with torch.no_grad():
            return self.model(data.to(self.device))

    def postprocess(self, data):
        return data.cpu().numpy().tolist()
"""

HANDLER_FILE = os.path.join(EXPORT_PATH, "handler.py")
with open(HANDLER_FILE, "w") as f:
    f.write(handler_code)

print(f"Handler written to {HANDLER_FILE}")

In [ ]:
!pip install -q torch-model-archiver

In [ ]:
# The prebuilt PyTorch containers require the archive to be named exactly model.mar,
# so the archiver's --model-name must be "model".
# !pip install -q torch-model-archiver

!torch-model-archiver \
    --model-name model \
    --version 1.0 \
    --serialized-file {SERIALIZED_FILE} \
    --handler {HANDLER_FILE} \
    --export-path {MODEL_STORE} \
    --force

!ls -la {MODEL_STORE}

In [ ]:
# ==============================================================================
# 1. Initialization and Constants
# ==============================================================================
PROJECT = !gcloud config get-value project 2>/dev/null
PROJECT = PROJECT[0]
PROJECT_ID = PROJECT
BUCKET = PROJECT
REGION = "us-central1"
MODEL_DISPLAYNAME = f"rl-ppo-rllib-{TIMESTAMP}"

print(f"MODEL_DISPLAYNAME: {MODEL_DISPLAYNAME}")

os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

In [ ]:
%%bash
# Create GCS bucket if it doesn't exist already...
exists=$(gsutil ls -d | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
else
    echo "Creating a new GCS bucket."
    gsutil mb -l ${REGION} gs://${BUCKET}
    echo "\nHere are your current buckets:"
    gsutil ls
fi

In [ ]:
# model.mar must sit directly under the artifact URI directory.
!gsutil cp {MODEL_STORE}/model.mar gs://{BUCKET}/{MODEL_DISPLAYNAME}/model.mar
!gsutil ls gs://{BUCKET}/{MODEL_DISPLAYNAME}/

## Create and Deploy to an Endpoint

After uploading, `model.deploy` creates an endpoint and deploys the model to it. An endpoint is a dedicated Vertex AI resource for serving predictions. Parameters include `machine_type` (machine resources) and `accelerator_type`/`accelerator_count` (for GPU acceleration).

The serving container changes from the TensorFlow image to a **PyTorch** one, since we are now shipping a TorchServe archive rather than a SavedModel. Container tags are versioned and retired on a schedule — check the [prebuilt containers list](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers#pytorch) and pick a currently supported PyTorch tag if the one below has aged out.

**The deployment takes around 10 minutes.**

In [ ]:
from google.cloud import aiplatform

# ==============================================================================
# 1. Initialization and Constants
# ==============================================================================
BUCKET_URI = f"gs://{BUCKET}/{MODEL_DISPLAYNAME}"

# Prebuilt PyTorch (TorchServe) prediction container. Verify the tag is current:
# https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers#pytorch
TORCH_PREDICTION_CONTAINER = "us-docker.pkg.dev/vertex-ai/prediction/pytorch-cpu.2-4:latest"

print("📌 Initializing Vertex AI SDK...")
aiplatform.init(project=PROJECT_ID, location=REGION)

# ==============================================================================
# 2. Register Model inside the Model Registry
# ==============================================================================
print("📌 Registering model in Vertex AI Model Registry...")
model = aiplatform.Model.upload(
    display_name="building-control-ppo-rllib-model-v2",
    artifact_uri=BUCKET_URI,
    serving_container_image_uri=TORCH_PREDICTION_CONTAINER,
    description="Ray RLlib PPO policy for Smart Building thermal control.",
)
print("✔️ Model registered successfully!")
print(f"   Model Resource Name: {model.resource_name}")

# ==============================================================================
# 3. Create the Serving Endpoint
# ==============================================================================
print("📌 Creating managed Vertex AI Endpoint...")
endpoint = aiplatform.Endpoint.create(
    display_name="building-control-ppo-endpoint-v2",
    description="Live HTTP endpoint for real-time temperature control policies.",
)
print("✔️ Endpoint created successfully!")
print(f"   Endpoint Resource Name: {endpoint.resource_name}")



# ==============================================================================
# 4. Deploy the Model to the Endpoint
# ==============================================================================
print("📌 Deploying model to Endpoint (this might take 3 to 10 minutes)...")
deployed_model = model.deploy(
    endpoint=endpoint,
    deployed_model_display_name="building-control-ppo-rllib-v1",
    machine_type="n1-standard-2",  # Cost-effective default instance
    min_replica_count=1,           # Minimum nodes for high-availability
    max_replica_count=1,           # Max node scaling limit under high traffic
)
print("\n🎉 Success! Your model is now fully deployed and serving live traffic.")
print(f"👉 Public Endpoint API URL: https://{REGION}-aiplatform.googleapis.com/v1/{endpoint.resource_name}:predict")

In [ ]:
ENDPOINT_ID = endpoint.resource_name

Once deployed, the model is accessible for online prediction requests.

Unlike the TensorFlow SavedModel path, TorchServe has no named serving signature — the handler defines the contract. Ours takes each instance as a plain 4-element list `[norm_indoor, norm_outdoor, norm_pressure, last_action]` and returns a 1-element list holding the clamped HVAC action, so `predictions[i][0]` is the number we want.

In [ ]:
import subprocess

import google.auth

# ==============================================================================
# 1. PROGRAMMATIC PROJECT AUTO-DISCOVERY & SDK INITIALIZATION
# ==============================================================================
print("🔄 Step 1: Programmatically auto-detecting GCP environment...")

try:
    _, detected_project_id = google.auth.default()
except Exception:
    detected_project_id = None

if not detected_project_id:
    try:
        detected_project_id = (
            subprocess.check_output("gcloud config get-value project", shell=True)
            .decode()
            .strip()
        )
    except Exception:
        detected_project_id = PROJECT

LOCATION = "us-central1"

print(f"   👉 Detected GCP Project ID: '{detected_project_id}'")
print(f"   👉 Servicing Region:         '{LOCATION}'")

aiplatform.init(project=detected_project_id, location=LOCATION)

# ==============================================================================
# 2. ROBUST VERTEX AI ENDPOINT FORCE-LOADING
# ==============================================================================
print("\n🔄 Step 2: Querying active endpoint metadata...")
endpoints = aiplatform.Endpoint.list(filter='display_name="building-control-ppo-endpoint"')

if not endpoints:
    endpoints = aiplatform.Endpoint.list()

if endpoints:
    numeric_id = endpoints[0].name
    print(f"   👉 Found active Endpoint: '{endpoints[0].display_name}'")
    print(f"   👉 Numeric Endpoint ID:   '{numeric_id}'")
    endpoint = aiplatform.Endpoint(endpoint_name=numeric_id)
    print("   ✔️ Connection successfully established and synced!")
else:
    raise ValueError(
        f"❌ No deployed endpoints found in project '{detected_project_id}'. "
        f"Please verify that your deployment cell has completed successfully."
    )

# ==============================================================================
# 3. EVALUATION SIMULATION RUN
# ==============================================================================
TEST_STEPS = 2000
print(f"\n🔄 Step 3: Starting Online Inference Evaluation Loop ({TEST_STEPS} steps)...")

test_env = BuildingControlEnv(
    {
        "data": test_data_norm,
        "max_steps": TEST_STEPS,
        "temp_mean": 20.0,
        "temp_std": 10.0,
        "start_idx": 0,
    }
)
state, _ = test_env.reset()

indoor_log = []
outdoor_log = []
action_log = []

for t in range(TEST_STEPS):
    if (t + 1) % 100 == 0 or t == 0:
        print(f"   [Step {t+1:04d}/{TEST_STEPS}] Querying optimal action from Vertex AI...")

    # A. PAYLOAD: the observation as a plain list of floats.
    state_list = np.asarray(state, dtype=np.float32).tolist()
    instances = [{"data": state_list}]

    # B. CALL DEPLOYED MODEL ENDPOINT
    response = endpoint.predict(instances=instances)

    # C. PARSE: the handler returns [[action]] per instance.
    prediction = response.predictions[0]
    action = float(prediction[0]) if isinstance(prediction, list) else float(prediction)

    # D. SIMULATE STEP (Gymnasium five-tuple)
    state, reward, terminated, truncated, info = test_env.step(action)

    # E. RECORD DATA & DENORMALIZE OUTDOOR TEMP
    indoor_log.append(test_env.indoor)
    outdoor_log.append((float(state[1]) * temp_std) + temp_mean)
    action_log.append(action)

    if terminated or truncated:
        state, _ = test_env.reset()

print("\n🎉 Step 3 Complete: All physical thermal metrics have been successfully logged!")

# ==============================================================================
# 4. DATA VISUALIZATION
# ==============================================================================
print("\n🔄 Step 4: Generating publication-grade performance charts...")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

ax1.plot(indoor_log, label="Indoor Temp (PPO Controlled)", color="crimson", linewidth=2)
ax1.plot(outdoor_log, label="Outdoor Ambient Temp", color="royalblue", alpha=0.5, linestyle="--")
ax1.axhline(y=TARGET_TEMP, color="forestgreen", linestyle=":", label="Comfort Target (22°C)")
ax1.set_ylabel("Temperature (°C)", fontsize=12)
ax1.set_title(
    "Smart Building HVAC Control: Vertex AI Online Serving Evaluation",
    fontsize=14,
    fontweight="bold",
)
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper right", frameon=True, facecolor="white")

ax2.plot(action_log, label="HVAC Action (Effort)", color="darkorange", linewidth=1.5, alpha=0.8)
ax2.set_ylabel("Control Effort [Cooling -1.0 to Heating +1.0]", fontsize=12)
ax2.set_xlabel("Simulation Timesteps", fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.legend(loc="upper right", frameon=True, facecolor="white")

plt.tight_layout()
plt.show()

print("🎉 Success! Your model evaluation is complete.")

## Cleanup

When deploying a model to an endpoint for online prediction, the minimum `min-replica-count` is 1, and it is charged per node hour. So let's delete the endpoint to reduce unnecessary charges. Before we can delete the endpoint, we first undeploy all attached models.

We also stop the RLlib algorithm and shut down Ray, which terminates the `EnvRunner` worker processes.

In [ ]:
#endpoint.undeploy_all()
#endpoint.delete()

algo.stop()
ray.shutdown()
print("Ray shut down.")

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.